# IGCRN: T4 + Google Drive
Select Runtime → Change runtime type → T4 GPU. No research results are bundled. Download/extract the public corpora according to the README before simulation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
%cd /content
!git clone https://github.com/ZhengchenYuan/IGCRN-DualMic-Speech-Enhancement.git
%cd /content/IGCRN-DualMic-Speech-Enhancement
%pip install -r requirements.txt
%pip install -e .


In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable T4 GPU first'
print(torch.cuda.get_device_name(0))
!nvidia-smi
!pytest -q
!python -m scripts.overfit


Extract LibriSpeech and MUSAN to `/content/LibriSpeech` and `/content/musan`. These cells deliberately stop if data is missing. Keep the same frozen mixtures for both runs.

In [ ]:
!mkdir -p /content/igcrn_data
!python -m scripts.catalog --librispeech /content/LibriSpeech --musan /content/musan --output /content/igcrn_data/catalog.jsonl
!python -m scripts.simulate --catalog /content/igcrn_data/catalog.jsonl --output /content/igcrn_data/mixtures --count 300 --seconds 2 --seed 2026


In [ ]:
!mkdir -p /content/drive/MyDrive/IGCRN/manifests
!cp /content/igcrn_data/mixtures/*.jsonl /content/drive/MyDrive/IGCRN/manifests/
!pip freeze > /content/drive/MyDrive/IGCRN/environment.txt
!git rev-parse HEAD > /content/drive/MyDrive/IGCRN/commit.txt


In [ ]:
!python -m scripts.train --config configs/small.yaml --data /content/igcrn_data/mixtures --mode dual --output /content/drive/MyDrive/IGCRN/dual --batch-size 1
!python -m scripts.train --config configs/small.yaml --data /content/igcrn_data/mixtures --mode duplicate --output /content/drive/MyDrive/IGCRN/duplicate --batch-size 1


If interrupted, add `--resume /content/drive/MyDrive/IGCRN/<arm>/last.pt` to that arm’s command. Regenerate data only with identical sources, offsets, seed and environment; verify WAV hashes against saved manifests. Do not overwrite the saved manifests until verified.

In [ ]:
!python -m scripts.evaluate --manifest /content/igcrn_data/mixtures/test.jsonl --dual /content/drive/MyDrive/IGCRN/dual/last.pt --duplicate /content/drive/MyDrive/IGCRN/duplicate/last.pt --output /content/drive/MyDrive/IGCRN/evaluation
